# Train from expanded crops

**Run All** neste notebook. Não precisa achar célula no `01-...`.

Pré-requisito: tabela de crops gerada pelo expand:

```text
python scripts/expand_lidc_dataset.py run --batch-size 25 --max-gb 8
```

Este notebook só chama `scripts/train_from_crops.py` (semantic + image + fusion).

In [1]:
from pathlib import Path
import subprocess
import sys

ROOT = Path("..").resolve()
if not (ROOT / "scripts" / "train_from_crops.py").exists():
    ROOT = Path(".").resolve()

script = ROOT / "scripts" / "train_from_crops.py"
crops_csv = ROOT / "outputs" / "preprocessing" / "lidc_model_table_fixed_crops.csv"

print("ROOT:", ROOT)
print("script:", script)
print("crops_csv exists:", crops_csv.exists())
if crops_csv.exists():
    import pandas as pd
    df = pd.read_csv(crops_csv)
    print(f"nodules={len(df)} patients={df['patient_id'].nunique()}")
    print(df["label"].value_counts())
else:
    raise FileNotFoundError(
        "Rode antes: python scripts/expand_lidc_dataset.py run --batch-size 25 --max-gb 8"
    )

ROOT: C:\Users\Administrator\Documents\Biopark\Projeto-Integrador-3\PI3-Grupo-03
script: C:\Users\Administrator\Documents\Biopark\Projeto-Integrador-3\PI3-Grupo-03\scripts\train_from_crops.py
crops_csv exists: True
nodules=27 patients=12
label
0    16
1    11
Name: count, dtype: int64


In [2]:
# Treina os 3 baselines. Ajuste --models se quiser só um, ex: --models fusion
cmd = [
    sys.executable,
    str(script),
    "--models", "semantic,image,fusion",
    "--epochs", "100",
    "--patience", "20",
]
print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, cwd=str(ROOT))
if proc.returncode != 0:
    raise RuntimeError(f"train_from_crops.py failed with exit code {proc.returncode}")

Running: c:\Users\Administrator\Documents\Biopark\Projeto-Integrador-3\PI3-Grupo-03\.venv\Scripts\python.exe C:\Users\Administrator\Documents\Biopark\Projeto-Integrador-3\PI3-Grupo-03\scripts\train_from_crops.py --models semantic,image,fusion --epochs 100 --patience 20


In [3]:
import pandas as pd

rows = []
for path in [
    ROOT / "models" / "semantic_mlp" / "semantic_only_mlp_results.csv",
    ROOT / "models" / "image_cnn" / "image_only_cnn_results.csv",
    ROOT / "models" / "fusion_cnn_mlp" / "fusion_cnn_mlp_test_results.csv",
]:
    if path.exists():
        part = pd.read_csv(path)
        rows.append(part)
        print("\n", path.name)
        display(part)

print("\nWeights:")
for p in [
    ROOT / "models" / "semantic_mlp" / "best_semantic_mlp.pth",
    ROOT / "models" / "image_cnn" / "best_image_cnn.pth",
    ROOT / "models" / "fusion_cnn_mlp" / "best_fusion_cnn_mlp.pth",
]:
    print(" ", p.name, "OK" if p.exists() else "MISSING")


 semantic_only_mlp_results.csv


,model,accuracy,auc,precision,sensitivity,specificity,f1,tn,fp,fn,tp
0,Semantic-only MLP,0.0,NaN,0.0,0.0,0.0,0.0,0,0,3,0



 image_only_cnn_results.csv


,model,accuracy,auc,precision,sensitivity,specificity,f1,tn,fp,fn,tp
0,Image-only CNN,1.0,NaN,1.0,1.0,0.0,1.0,0,0,0,3



 fusion_cnn_mlp_test_results.csv


,model,threshold_type,threshold,accuracy,auc,precision,sensitivity,specificity,f1,tn,fp,fn,tp
0,Fusion CNN-MLP,default_0.5,0.500000,0.0,NaN,0.0,0.0,0.0,0.0,0,0,3,0
1,Fusion CNN-MLP,validation_youden_j,0.481153,1.0,NaN,1.0,1.0,0.0,1.0,0,0,0,3



Weights:
  best_semantic_mlp.pth OK
  best_image_cnn.pth OK
  best_fusion_cnn_mlp.pth OK
